# scPyviewer API Tutorial

**scPyviewer** is a Python-native viewer and programmatic API for single-cell AnnData (`.h5ad`) objects.  
This notebook walks through every API function with runnable examples.

## Installation

```bash
pip install scPyviewer          # core (no Streamlit)
pip install 'scPyviewer[all]'   # core + viewer + clustering + xlsx export
```

## Contents

1. [Setup & toy dataset](#1-setup)
2. [Load a dataset](#2-load)
3. [Global style](#3-style)
4. [`plot_embedding`](#4-embedding) — UMAP / t-SNE scatter
5. [`plot_multigene`](#5-multigene) — multi-gene expression grid
6. [`plot_violin`](#6-violin) — violin / box plot
7. [`plot_dotplot`](#7-dotplot) — dot plot
8. [`plot_composition`](#8-composition) — stacked bar
9. [Table functions](#9-tables)
10. [Export figures & tables](#10-export)
11. [Publication-quality workflow](#11-publication)

---
## 1 · Setup & toy dataset <a id="1-setup"></a>

We build a small synthetic AnnData so the notebook runs without downloading data.  
Replace `tmp_path` with your own `*.prepared.h5ad` when working with real data.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import tempfile, os
import matplotlib.pyplot as plt

# ── build a synthetic 200-cell × 80-gene AnnData ──────────────────────────
rng = np.random.default_rng(42)
n_cells, n_genes = 200, 80
gene_names = [f"Gene{i:03d}" for i in range(n_genes)]

counts = (
    sp.random(n_cells, n_genes, density=0.35, format="csr", random_state=42)
    .toarray() * 12
).astype(np.float32)
lognorm = np.log1p(
    counts / np.clip(counts.sum(axis=1, keepdims=True), 1, None) * 1e4
).astype(np.float32)

cell_types = np.array(
    ["T cell"] * 60 + ["B cell"] * 50 + ["NK cell"] * 40 + ["Monocyte"] * 50
)
batch = np.array(["Sample_A"] * 100 + ["Sample_B"] * 100)

obs = pd.DataFrame({
    "cell_type": pd.Categorical(cell_types),
    "batch":     pd.Categorical(batch),
    "n_counts":  counts.sum(axis=1),
})

adata = ad.AnnData(
    X=sp.csr_matrix(lognorm),
    obs=obs,
    var=pd.DataFrame(index=gene_names),
)
adata.layers["lognorm"] = sp.csr_matrix(lognorm)

# fake UMAP and t-SNE
adata.obsm["X_umap"] = rng.standard_normal((n_cells, 2)).astype(np.float32)
adata.obsm["X_tsne"] = rng.standard_normal((n_cells, 2)).astype(np.float32)

# synthetic marker/DE table (scPyviewer_markers format)
rows = []
for g_idx, group in enumerate(["T cell", "B cell", "NK cell", "Monocyte"]):
    for rank in range(1, 11):
        gene = gene_names[(g_idx * 10 + rank - 1) % n_genes]
        rows.append(dict(group=group, rank=rank, gene=gene,
                         logfoldchange=float(rng.uniform(0.5, 3.0)),
                         score=float(rng.uniform(1, 10)),
                         pval=float(rng.uniform(0, 0.05)),
                         pval_adj=float(rng.uniform(0, 0.1))))
mdf = pd.DataFrame(rows)
adata.uns["scPyviewer_markers"] = {c: mdf[c].tolist() for c in mdf.columns}
adata.uns["scPyviewer"] = {
    "group_key": "cell_type",
    "embeddings": ["X_umap", "X_tsne"],
    "schema": {
        "categorical_obs": ["cell_type", "batch"],
        "numeric_obs": ["n_counts"],
    },
    "version": "0.3.0",
}

# save to a temp file (mimics a real .prepared.h5ad)
tmp_path = os.path.join(tempfile.mkdtemp(), "toy.prepared.h5ad")
adata.write_h5ad(tmp_path)
print("Toy dataset written to:", tmp_path)

---
## 2 · Load a dataset <a id="2-load"></a>

In [ ]:
import scPyviewer as sv

print("scPyviewer version:", sv.__version__)

# load_dataset() reads a *.prepared.h5ad and returns a Dataset handle
ds = sv.load_dataset(tmp_path)
print(ds)

In [ ]:
# inspect the dataset
print("Cells      :", ds.n_obs)
print("Genes      :", ds.n_vars)
print("Group key  :", ds.group_key)       # primary clustering column
print("Embeddings :", ds.embeddings)      # available 2-D projections
print("Categorical:", ds.categorical)     # obs columns available for grouping

# search for genes (substring match)
print("\nGenes matching '01':", ds.genes("01"))

---
## 3 · Global style <a id="3-style"></a>

`sv.set_style()` sets matplotlib defaults for **all subsequent** `plot_*` calls.  
Individual functions can still override any setting via their own parameters.

In [ ]:
# set a clean global style for the whole notebook
sv.set_style(
    font_family="DejaVu Sans",   # try "Arial", "Helvetica", "Times New Roman"
    base_fontsize=10,
    dpi=120,
)

# available matplotlib style sheets:
# sv.set_style(style="seaborn-v0_8-whitegrid")
# sv.set_style(style="ggplot")
print("Global style applied.")

---
## 4 · `plot_embedding` <a id="4-embedding"></a>

Scatter plot of any 2-D embedding (UMAP, t-SNE, PCA …), colored by a metadata column or gene expression.

In [ ]:
# ── 4.1  default: color by cell type ─────────────────────────────────────
fig = sv.plot_embedding(ds, color="cell_type")
plt.show()

In [ ]:
# ── 4.2  color by gene expression ────────────────────────────────────────
fig = sv.plot_embedding(ds, gene="Gene000")
plt.show()

In [ ]:
# ── 4.3  t-SNE instead of UMAP ───────────────────────────────────────────
fig = sv.plot_embedding(ds, color="cell_type", embedding="X_tsne")
plt.show()

In [ ]:
# ── 4.4  full typography control ─────────────────────────────────────────
fig = sv.plot_embedding(
    ds,
    color="cell_type",
    # layout
    figsize=(7, 5.5),
    dpi=150,
    # scatter
    point_size=6,
    alpha=0.85,
    # labels
    title="Immune cell types — UMAP",
    title_fontsize=14,
    label_groups=True,
    label_fontsize=9,
    xlabel="UMAP 1",
    ylabel="UMAP 2",
    xlabel_fontsize=11,
    ylabel_fontsize=11,
    tick_fontsize=9,
    # legend (shown when label_groups=False)
    show_legend=False,
    legend_fontsize=9,
    # font
    font_family="DejaVu Sans",
)
plt.show()

In [ ]:
# ── 4.5  gene expression with colorbar customization ─────────────────────
fig = sv.plot_embedding(
    ds,
    gene="Gene010",
    figsize=(6, 5),
    dpi=150,
    cmap="magma",
    point_size=8,
    title="Gene010 expression",
    title_fontsize=13,
    colorbar_label="log-norm expr",
    colorbar_fontsize=10,
)
plt.show()

---
## 5 · `plot_multigene` <a id="5-multigene"></a>

Grid of embedding panels, one per gene.

In [ ]:
# ── 5.1  default ─────────────────────────────────────────────────────────
top_genes = [f"Gene{i:03d}" for i in range(6)]
fig = sv.plot_multigene(ds, genes=top_genes)
plt.show()

In [ ]:
# ── 5.2  2-column layout with custom typography ───────────────────────────
fig = sv.plot_multigene(
    ds,
    genes=top_genes,
    ncol=2,                      # 2-column grid
    figsize=(7, 10),             # override auto-size
    dpi=150,
    point_size=4,
    cmap="RdBu_r",
    suptitle="Top marker genes",
    suptitle_fontsize=14,
    panel_title_fontsize=11,
    colorbar_fontsize=8,
    font_family="DejaVu Sans",
)
plt.show()

---
## 6 · `plot_violin` <a id="6-violin"></a>

Violin or box plot of gene expression per group.

In [ ]:
# ── 6.1  default violin ───────────────────────────────────────────────────
fig = sv.plot_violin(ds, gene="Gene000", group="cell_type")
plt.show()

In [ ]:
# ── 6.2  box plot with data points ────────────────────────────────────────
fig = sv.plot_violin(
    ds,
    gene="Gene000",
    group="cell_type",
    kind="box",
    show_points=True,
)
plt.show()

In [ ]:
# ── 6.3  publication-quality typography ──────────────────────────────────
fig = sv.plot_violin(
    ds,
    gene="Gene000",
    group="cell_type",
    kind="violin",
    show_points=True,
    # layout
    figsize=(8, 4.5),
    dpi=150,
    # typography
    title="Gene000 expression by cell type",
    title_fontsize=14,
    xlabel="Cell type",
    ylabel="Log-normalized expression",
    xlabel_fontsize=12,
    ylabel_fontsize=12,
    tick_fontsize=10,
    rotation=20,
    font_family="DejaVu Sans",
)
plt.show()

In [ ]:
# ── 6.4  split by batch ───────────────────────────────────────────────────
fig = sv.plot_violin(ds, gene="Gene005", group="batch")
plt.show()

---
## 7 · `plot_dotplot` <a id="7-dotplot"></a>

Dot size = fraction of cells expressing the gene.  
Dot color = mean log-normalized expression.

In [ ]:
# ── 7.1  default ─────────────────────────────────────────────────────────
marker_genes = [f"Gene{i:03d}" for i in [0, 10, 20, 30, 40]]
fig = sv.plot_dotplot(ds, genes=marker_genes, group="cell_type")
plt.show()

In [ ]:
# ── 7.2  scale within each gene (standard_scale="var") ───────────────────
fig = sv.plot_dotplot(
    ds,
    genes=marker_genes,
    group="cell_type",
    standard_scale="var",   # normalize 0-1 per gene column
    cmap="Blues",
)
plt.show()

In [ ]:
# ── 7.3  full typography control ─────────────────────────────────────────
fig = sv.plot_dotplot(
    ds,
    genes=marker_genes,
    group="cell_type",
    standard_scale="var",
    cmap="viridis",
    size_scale=250,
    # layout
    figsize=(7, 3.5),
    dpi=150,
    # typography
    title="Marker gene expression",
    title_fontsize=13,
    xlabel="Gene",
    ylabel="Cell type",
    xlabel_fontsize=11,
    ylabel_fontsize=11,
    tick_fontsize=9,
    gene_label_rotation=30,
    colorbar_label="Scaled mean expr",
    colorbar_fontsize=9,
    legend_fontsize=8,
    font_family="DejaVu Sans",
)
plt.show()

---
## 8 · `plot_composition` <a id="8-composition"></a>

Stacked bar showing cell-type proportions across samples / conditions.

In [ ]:
# ── 8.1  default (fraction) ───────────────────────────────────────────────
fig = sv.plot_composition(ds, group="cell_type", split="batch")
plt.show()

In [ ]:
# ── 8.2  raw cell counts ─────────────────────────────────────────────────
fig = sv.plot_composition(ds, group="cell_type", split="batch", normalize=False)
plt.show()

In [ ]:
# ── 8.3  full typography control ─────────────────────────────────────────
fig = sv.plot_composition(
    ds,
    group="cell_type",
    split="batch",
    normalize=True,
    sort_groups=True,
    # layout
    figsize=(6, 4),
    dpi=150,
    bar_width=0.65,
    # typography
    title="Cell-type composition per sample",
    title_fontsize=13,
    xlabel="Sample",
    ylabel="Fraction of cells",
    xlabel_fontsize=11,
    ylabel_fontsize=11,
    tick_fontsize=9,
    rotation=0,
    legend_fontsize=8,
    legend_title_fontsize=9,
    font_family="DejaVu Sans",
)
plt.show()

---
## 9 · Table functions <a id="9-tables"></a>

In [ ]:
# ── 9.1  marker / DE table ───────────────────────────────────────────────
mk = sv.markers_table(ds, top_n=5)      # top 5 genes per group
print(f"Shape: {mk.shape}")
mk.head(10)

In [ ]:
# filter to a single cell type
t_markers = sv.markers_table(ds, group="T cell", top_n=5)
t_markers

In [ ]:
# sort by logfoldchange instead of rank
sv.markers_table(ds, sort_by="logfoldchange", ascending=False, top_n=5)

In [ ]:
# ── 9.2  composition table ────────────────────────────────────────────────
comp = sv.composition_table(ds, group="cell_type", split="batch")
comp   # fraction of each cell type per batch

In [ ]:
# ── 9.3  metadata table ───────────────────────────────────────────────────
meta = sv.metadata_table(ds)
print(f"Shape: {meta.shape}")
meta.head()

---
## 10 · Export figures & tables <a id="10-export"></a>

In [ ]:
# ── 10.1  export all standard figures ─────────────────────────────────────
fig_dir = "/tmp/scpyviewer_figs"

written = sv.export_figures(
    ds,
    outdir=fig_dir,
    formats=["png", "pdf"],      # any of: png, pdf, svg
    dpi=200,
    # apply consistent typography to all exported figures
    title_fontsize=12,
    tick_fontsize=9,
    label_fontsize=10,
    font_family="DejaVu Sans",
)

print("Written files:")
for p in written:
    print(" ", p)

In [ ]:
# ── 10.2  save a single figure manually ──────────────────────────────────
fig = sv.plot_embedding(ds, color="cell_type", dpi=200, figsize=(7, 5))
fig.savefig("/tmp/umap_celltypes.png", dpi=200, bbox_inches="tight")
fig.savefig("/tmp/umap_celltypes.pdf",           bbox_inches="tight")
fig.savefig("/tmp/umap_celltypes.svg",           bbox_inches="tight")
print("Saved PNG / PDF / SVG")

In [ ]:
# ── 10.3  export tables ───────────────────────────────────────────────────
tbl_dir = "/tmp/scpyviewer_tables"

written = sv.export_tables(
    ds,
    outdir=tbl_dir,
    formats=["csv", "tsv"],   # add "xlsx" if openpyxl is installed
    top_n=25,
)

print("Written tables:")
for p in written:
    print(" ", p)

---
## 11 · Publication-quality workflow <a id="11-publication"></a>

A complete example: global style → per-figure fine-tuning → batch export at 300 DPI.

In [ ]:
# ── Step 1: apply a global style for the whole analysis ───────────────────
sv.set_style(
    font_family="DejaVu Sans",
    base_fontsize=11,
    dpi=300,
)

In [ ]:
# ── Step 2: UMAP overview for panel A ─────────────────────────────────────
fig_a = sv.plot_embedding(
    ds,
    color="cell_type",
    figsize=(5.5, 4.5),
    point_size=5, alpha=0.8,
    title="A  Cell-type annotation",
    title_fontsize=12,
    label_fontsize=8,
    xlabel_fontsize=10, ylabel_fontsize=10,
    tick_fontsize=8,
)
plt.show()

In [ ]:
# ── Step 3: dot plot for panel B ──────────────────────────────────────────
fig_b = sv.plot_dotplot(
    ds,
    genes=[f"Gene{i:03d}" for i in [0, 10, 20, 30, 40, 50]],
    group="cell_type",
    standard_scale="var",
    cmap="Blues",
    figsize=(7, 3),
    title="B  Marker gene expression",
    title_fontsize=12,
    tick_fontsize=9,
    colorbar_fontsize=9,
    legend_fontsize=8,
    gene_label_rotation=35,
)
plt.show()

In [ ]:
# ── Step 4: composition bar for panel C ───────────────────────────────────
fig_c = sv.plot_composition(
    ds,
    group="cell_type", split="batch",
    figsize=(4.5, 4),
    title="C  Sample composition",
    title_fontsize=12,
    xlabel_fontsize=10, ylabel_fontsize=10,
    tick_fontsize=9, rotation=0,
    legend_fontsize=8,
)
plt.show()

In [ ]:
# ── Step 5: batch-export all panels at 300 DPI ────────────────────────────
out = "/tmp/paper_figs"
os.makedirs(out, exist_ok=True)

for name, fig in [("panelA_umap", fig_a),
                  ("panelB_dotplot", fig_b),
                  ("panelC_composition", fig_c)]:
    for fmt in ("png", "pdf", "svg"):
        path = os.path.join(out, f"{name}.{fmt}")
        fig.savefig(path, dpi=300, bbox_inches="tight")

print("Exported to", out)
print(os.listdir(out))

---
## Parameter quick-reference

| Parameter | Type | Applies to | What it controls |
|---|---|---|---|
| `figsize` | `(float, float)` | all | Figure width × height in inches |
| `dpi` | `int` | all | Resolution (dots per inch) |
| `title` | `str` | all | Figure title text |
| `title_fontsize` | `float` | all | Title font size |
| `xlabel` / `ylabel` | `str` | violin, dotplot, composition | Axis label text |
| `xlabel_fontsize` / `ylabel_fontsize` | `float` | violin, dotplot, composition | Axis label font size |
| `tick_fontsize` | `float` | all | Tick-label font size |
| `legend_fontsize` | `float` | embedding, dotplot, composition | Legend entry font size |
| `colorbar_fontsize` | `float` | embedding (gene), multigene, dotplot | Colorbar text size |
| `label_fontsize` | `float` | embedding | Centroid text label size |
| `panel_title_fontsize` | `float` | multigene | Per-panel title size |
| `suptitle_fontsize` | `float` | multigene | Figure super-title size |
| `gene_label_rotation` | `int` | dotplot | X-axis gene label rotation (°) |
| `rotation` | `int` | violin, composition | X-tick rotation (°) |
| `font_family` | `str` | all | Per-figure font family |
| `point_size` | `float` | embedding, multigene | Scatter point diameter |
| `alpha` | `float` | embedding, multigene | Point transparency |
| `cmap` | `str` | embedding, multigene, dotplot | Matplotlib colormap |
| `kind` | `str` | violin | `"violin"` or `"box"` |
| `show_points` | `bool` | violin | Overlay jittered data points |
| `standard_scale` | `str` | dotplot | `None`, `"var"`, or `"group"` |
| `normalize` | `bool` | composition | Fraction vs. raw counts |

---
*Generated with scPyviewer. See [github.com/xuan13hao/scPyviewer](https://github.com/xuan13hao/scPyviewer) for the source.*